# Titans / Miras Parametric Memory — Standalone Demo

This notebook walks through the **neural long-term memory** module that
lives in `rlm.memory`, with no large LM in the loop.  We show:

1. Building a `TitansMemory` module.
2. Streaming `(key, value)` pairs through it and reading the memory.
3. Switching to a `MirasMemory` with different retention losses
   (`l2`, `l1`, `huber`, `kl`).
4. A synthetic associative-recall task that exercises the memory
   the way the long-context "needle in haystack" benchmarks do.

The whole notebook runs in <30s on CPU.


In [ ]:
# === Install ============================================================
# Colab usually has torch.  We install rlms from this repo (editable) so the
# notebook uses the same memory module that the rest of the library uses.
%pip install --quiet torch
import os
if not os.path.exists("/content/rlm"):
    !git clone --depth 1 https://github.com/alexzhang13/rlm /content/rlm 2>/dev/null || true
%pip install --quiet -e /content/rlm 2>/dev/null || %pip install --quiet -e .


In [ ]:
import torch
from rlm.memory import MemoryConfig, TitansMemory, MirasMemory

torch.manual_seed(0)
print("torch:", torch.__version__)


## 1. Build a Titans memory module

`MemoryConfig` controls the geometry and update rule.  Here we use a
small MLP (16 → 32 → 16) and disable the learnable gates so the inner
update is plain SGD.

In [ ]:
cfg = MemoryConfig(
    key_dim=16,
    value_dim=16,
    hidden_dim=32,
    n_layers=2,
    inner_lr=2.0,
    momentum=0.0,
    forget_rate=0.0,
    learnable_gates=False,
    chunk_size=1,
)
mem = TitansMemory(cfg)
print(mem)
print("parameter count:", sum(p.numel() for p in mem.parameters()))


## 2. Associative recall

We generate 8 random `(k, v)` pairs and stream them through the memory.
After enough rehearsal passes the memory MLP "stores" the associations:
querying with `k` recovers `v`.

In [ ]:
keys = torch.randn(1, 8, 16)
values = torch.randn(1, 8, 16)

state = mem.init_state(1)
err_before = (mem.read(keys, state) - values).pow(2).mean().item()
print(f"MSE before any writes: {err_before:.3f}")

errors = []
for epoch in range(6):
    state = mem.write(keys, values, state)
    err = (mem.read(keys, state) - values).pow(2).mean().item()
    errors.append(err)
    print(f"  epoch {epoch+1:>2}  MSE = {err:.3f}")


In [ ]:
# Quick matplotlib plot of the convergence curve.
import matplotlib.pyplot as plt
plt.figure(figsize=(4, 3))
plt.plot([err_before] + errors, marker="o")
plt.xlabel("rehearsal pass")
plt.ylabel("reconstruction MSE")
plt.title("Titans memory learns the (k,v) associations online")
plt.grid(alpha=0.3)
plt.show()


## 3. Miras: swap the retention surrogate

`MirasMemory` is a drop-in subclass that exposes the retention loss as
a config knob.  We can compare `l2` (= Titans), `l1`, `huber`, `kl`.

In [ ]:
def run(retention: str, epochs: int = 6):
    torch.manual_seed(0)
    cfg = MemoryConfig(
        key_dim=16, value_dim=16, hidden_dim=32, n_layers=2,
        inner_lr=2.0, momentum=0.0, forget_rate=0.0,
        learnable_gates=False, chunk_size=1, retention=retention,
    )
    m = MirasMemory(cfg)
    keys = torch.randn(1, 8, 16)
    values = torch.randn(1, 8, 16)
    state = m.init_state(1)
    out = [(m.read(keys, state) - values).pow(2).mean().item()]
    for _ in range(epochs):
        state = m.write(keys, values, state)
        out.append((m.read(keys, state) - values).pow(2).mean().item())
    return out

curves = {r: run(r) for r in ["l2", "l1", "huber", "kl"]}

plt.figure(figsize=(5, 3.5))
for name, c in curves.items():
    plt.plot(c, marker="o", label=name)
plt.legend()
plt.xlabel("rehearsal pass")
plt.ylabel("reconstruction MSE")
plt.title("Miras retention surrogates")
plt.grid(alpha=0.3)
plt.show()


## 4. Learnable gates + momentum

The realistic Titans configuration uses data-dependent `(lr, momentum,
forget)` gates and momentum surprise.  The gates are gradients-trained
end-to-end with the rest of the model in a meta-learning loop; here we
just check that the forward pass runs cleanly with them on.

In [ ]:
cfg_full = MemoryConfig(
    key_dim=16, value_dim=16, hidden_dim=32, n_layers=2,
    inner_lr=1.0, momentum=0.9, forget_rate=0.05,
    learnable_gates=True, chunk_size=2,
)
m_full = TitansMemory(cfg_full)
keys = torch.randn(1, 16, 16)
values = torch.randn(1, 16, 16)
out, _ = m_full.read_write(keys, values, keys)
print("output:", out.shape, "finite?", torch.isfinite(out).all().item())

# Outer-loop differentiability check: gradient flows to the initial memory
# parameters even though the memory was updated *during* the forward pass.
loss = out.pow(2).mean()
loss.backward()
any_grad = any(p.grad is not None and p.grad.abs().sum() > 0 for p in m_full.parameters())
print("memory params receive outer-loop gradient:", any_grad)


### What to do next

Continue with the next notebook, `titans_gemma_demo.ipynb`, where we
wrap the same memory module around a tiny **Gemma 3 1B-IT** to build a
memory-augmented LM.  Then `titans_rlm_benchmark.ipynb` runs the four
benchmark scripts (reasoning, tool use, code gen, long context) and
plots the results.